In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "seed2012chimpanzee")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Broken_tool_exp1.csv")
complete_path_2 = os.path.join(original_data_pathway, "Broken_tool_exp2.csv")
complete_path_3 = os.path.join(original_data_pathway, "Broken_tool_exp3.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df3 = pd.read_csv(complete_path_3)

intial_import_list = [[df1, '1'],
                      [df2, '2'],
                      [df3, '3']]
for x,y in intial_import_list:
    x['study_id']="seed2012chimpanzee"
    x['experiment']=y


data_frames=[df1, df2,df3]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"name": "participant",
                      'age':'age_in_years',
                      'group':'group_original'}, inplace=True) ##standardize names for participants
    x['participant'] = x['participant'].str.rstrip() ##remove spaces
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")
df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

fulldf.dropna(subset=['participant'], inplace=True)
# fulldf.columns
fulldf.columns = fulldf.columns.str.replace(' ', '_', regex=True)

In [4]:
fulldf = fulldf[['study_id', 'experiment', 
                 'participant','age_in_years',  'sex', 'species',
                 'group_original',  'visible', 'aligned',
       'covered', 'no_occlusion',
       'no_occlusion_no_delay', 'negative', 'inductive', 'positive',]]
for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'seed2012chimpanzee_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'seed2012chimpanzee_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)